In [1]:
from backtesting import Backtest, Strategy
import talib
import pandas as pd
import yfinance as yf
import math

C:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [2]:
cl = yf.Ticker("CL=F")
data = cl.history('max')

In [41]:
def cps(series):
    return series['Calmar Ratio']*series['Sortino Ratio']/series['Sharpe Ratio']

In [31]:
class MA(Strategy):
    ma_window = 32
    def init(self):
        self.ma = self.I(talib.MA, self.data.Close, self.ma_window)

    def next(self):
        price = self.data.Close[-1]
        m1 = self.ma[-1]
        m2 = self.ma[-2]
        slope_pct = (self.ma[-1] - self.ma[-2]) / self.ma[-2]
        slope_threshold = math.tan(math.pi/10)
        slope_angle = math.atan(slope_pct * 100)

        if not self.position:
            # Added RSI rising check and cleaned up the slope check
            if slope_angle >= slope_threshold and price > m1:
                self.buy()

        # EXIT: Give the trade more 'breathing room'
        elif self.position:
            # 1. Trailing Stop: Update SL to the current MA value if it's higher than the old SL
            # 2. Relaxed Exit: Only exit if price closes significantly below MA or RSI is pinned at top
            if price < m1 * 0.98 or slope_pct<0: 
                self.position.close()

In [18]:
bt = Backtest(data, MA, cash=10_000)

In [42]:
stats = bt.optimize(
    ma_window = range(20,50,2),
    maximize = cps,
)

C:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\backtesting\backtesting.py:1624: RuntimeWarning: If you want to use multi-process optimization with `multiprocessing.get_start_method() == 'spawn'` (e.g. on Windows),set `backtesting.Pool = multiprocessing.Pool` (or of the desired context) and hide `bt.optimize()` call behind a `if __name__ == '__main__'` guard. Currently using thread-based paralellism, which might be slightly slower for non-numpy / non-GIL-releasing code. See https://github.com/kernc/backtesting.py/issues/1256
  output = _optimize_grid()


Backtest.optimize:   0%|          | 0/15 [00:00<?, ?it/s]

Backtest.run:   0%|          | 0/6380 [00:00<?, ?bar/s]

C:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\backtesting\backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)


Backtest.run:   0%|          | 0/6378 [00:00<?, ?bar/s]

C:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\backtesting\backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)


... (19 outputs left out)

Backtest.run:   0%|          | 0/6360 [00:00<?, ?bar/s]

C:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\backtesting\backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)


Backtest.run:   0%|          | 0/6368 [00:00<?, ?bar/s]

C:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\backtesting\backtesting.py:1545: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = self.run(**dict(zip(heatmap.index.names, best_params)))


In [43]:
stats

Start                     2000-08-23 00:00...
End                       2026-02-19 00:00...
Duration                   9311 days 01:00:00
Exposure Time [%]                    31.79688
Equity Final [$]                  25770.50568
Equity Peak [$]                   48759.35587
Return [%]                          157.70506
Buy & Hold Return [%]               115.33225
Return (Ann.) [%]                     3.79776
Volatility (Ann.) [%]                19.64397
CAGR [%]                              2.59517
Sharpe Ratio                          0.19333
Sortino Ratio                         0.28473
Calmar Ratio                          0.07512
Alpha [%]                                 NaN
Beta                                      NaN
Max. Drawdown [%]                   -50.55251
Avg. Drawdown [%]                    -6.83246
Max. Drawdown Duration     2512 days 00:00:00
Avg. Drawdown Duration      176 days 00:00:00
# Trades                                   72
Win Rate [%]                      

In [21]:
bt.plot()

GridPlot(id='p1906', ...)

In [44]:
data_1y = cl.history('1y')

In [45]:
bt1 = Backtest(data_1y, MA, cash = 10_000)

In [46]:
stats_1 = bt1.run()

Backtest.run:   0%|          | 0/221 [00:00<?, ?bar/s]

C:\Users\DELL\AppData\Local\Temp\ipykernel_29388\2830469873.py:1: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats_1 = bt1.run()


In [47]:
stats_1

Start                     2025-02-19 00:00...
End                       2026-02-19 00:00...
Duration                    365 days 00:00:00
Exposure Time [%]                    10.67194
Equity Final [$]                    8680.4998
Equity Peak [$]                       10000.0
Return [%]                            -13.195
Buy & Hold Return [%]                -0.79163
Return (Ann.) [%]                   -13.14644
Volatility (Ann.) [%]                14.84371
CAGR [%]                             -9.30766
Sharpe Ratio                         -0.88566
Sortino Ratio                        -0.92268
Calmar Ratio                         -0.70081
Alpha [%]                           -12.99812
Beta                                   0.2487
Max. Drawdown [%]                     -18.759
Avg. Drawdown [%]                     -18.759
Max. Drawdown Duration      252 days 00:00:00
Avg. Drawdown Duration      252 days 00:00:00
# Trades                                    1
Win Rate [%]                      